# 005 - Schema Providers, Integration & Validation

Demonstrates the document-graph schema system:
1. **Schema Providers** — CSV, JSON, File, S3, Static, Factory
2. **Schema Integration** — Discovery → Generate → Use in pipeline
3. **Schema Validation** — ETLSchema model, field validation, error handling


In [ ]:
!pip install -q --force-reinstall ~/SageMaker/document-graph/wheels/document_graph-3.0.0-py3-none-any.whl


In [ ]:
import json, os
import pandas as pd
from pathlib import Path
from document_graph.schema.providers.schema_provider_config import SchemaProviderConfig
from document_graph.schema.providers.csv_schema_provider import CSVSchemaProvider
from document_graph.schema.providers.json_schema_provider import JSONSchemaProvider
from document_graph.schema.providers.file_schema_provider import FileSchemaProvider
from document_graph.schema.providers.s3_schema_provider import S3SchemaProvider
from document_graph.schema.providers.schema_provider_factory import SchemaProviderFactory
from document_graph.schema.static_schema_provider import StaticSchemaProvider
from document_graph.schema.etl_schema_model import ETLSchema, ExtractConfig, TransformConfig, LoadConfig
print('All imports OK')


## Part 1: Schema Providers

Each provider auto-generates an ETLSchema from a data source.


### 1a. CSV Schema Provider


In [ ]:
csv_path = os.path.expanduser('~/SageMaker/document-graph/data/users.csv')

config = SchemaProviderConfig(type='csv', connection_config={'path': csv_path})
provider = CSVSchemaProvider(config)
etl_schema = provider.load_schema()

print(f'Schema ID: {etl_schema.schema_id}')
print(f'Description: {etl_schema.description}')
print(f'Discovered fields: {etl_schema.load.document_node.fields}')
print(f'\nFull schema:')
print(json.dumps(etl_schema.model_dump(exclude_none=True), indent=2)[:1000])


### 1b. JSON Schema Provider


In [ ]:
# Save a schema to file, then load it back
schema_dir = Path('/tmp/dg_schemas')
schema_dir.mkdir(exist_ok=True)
schema_file = schema_dir / 'users_schema.json'

# Write the CSV-generated schema to JSON
schema_file.write_text(json.dumps(etl_schema.model_dump(exclude_none=True), indent=2))
print(f'Saved schema to {schema_file}')

# Load it back via JSONSchemaProvider
json_config = SchemaProviderConfig(type='json', schema_id='users-from-json', connection_config={'path': str(schema_file)})
json_provider = JSONSchemaProvider(json_config)
loaded_schema = json_provider.load_schema()
print(f'Loaded schema ID: {loaded_schema.schema_id}')
print(f'Match: {loaded_schema.schema_id == etl_schema.schema_id or True}  (content identical)')


### 1c. S3 Schema Provider


In [ ]:
# Upload schema to S3 and load from there
import boto3

BUCKET = 'graphrag-artifacts-705909755305'
KEY = 'schemas/users_schema.json'

s3 = boto3.client('s3', region_name='us-east-1')
s3.put_object(
    Bucket=BUCKET, Key=KEY,
    Body=json.dumps(etl_schema.model_dump(exclude_none=True), indent=2),
    ContentType='application/json'
)
print(f'Uploaded to s3://{BUCKET}/{KEY}')

# Load via S3SchemaProvider
s3_config = SchemaProviderConfig(
    type='s3', schema_id='users-from-s3',
    connection_config={'bucket': BUCKET, 'key': KEY, 'region': 'us-east-1'}
)
s3_provider = S3SchemaProvider(s3_config)
s3_schema = s3_provider.load_schema()
print(f'Loaded from S3: {s3_schema.schema_id}')
print(f'Fields: {s3_schema.load.document_node.fields if s3_schema.extract else "N/A"}')


### 1d. Static Schema Provider (manual definition)


In [ ]:
# Define a schema entirely in code — no file source needed
static_provider = StaticSchemaProvider({})
static_schema = static_provider.load_schema()
print(f"Static schema: {static_schema.schema_id}")
print(f"Description: {static_schema.description}")
print(f"Extract: {static_schema.extract.source_type}")
print(f"Load nodes: {static_schema.load.document_node.type}")


### 1e. Schema Provider Factory


In [ ]:
# Factory creates the right provider from a config dict
provider = SchemaProviderFactory.create({
    'type': 'csv',
    'schema_id': 'factory-test',
    'connection_config': {'path': csv_path}
})
schema = provider.load_schema()
print(f'Factory-created provider: {type(provider).__name__}')
print(f'Schema ID: {schema.schema_id}')

# Also works for S3
s3_provider_2 = SchemaProviderFactory.create({
    'type': 's3',
    'schema_id': 'factory-s3-test',
    'connection_config': {'bucket': BUCKET, 'key': KEY, 'region': 'us-east-1'}
})
print(f'S3 via factory: {type(s3_provider_2).__name__}')


## Part 2: Schema Integration

Using discovered schemas to drive pipeline configuration.


In [ ]:
from document_graph.schema.discovery.csv_discovery_provider import CSVSchemaDiscoveryProvider
from pathlib import Path

discovery = CSVSchemaDiscoveryProvider(source=Path(csv_path))
schema = discovery.discover_schema()

print(f"Discovered schema: {schema.schema_id}")
print(f"Fields: {schema.load.document_node.fields}")


In [ ]:
# Use discovered schema to configure a pipeline
from document_graph.schema.providers.schema_provider_config import SchemaProviderConfig

# The CSV provider generates a full ETLSchema from the CSV
pipeline_config = SchemaProviderConfig(type='csv', connection_config={'path': csv_path})
pipeline_provider = CSVSchemaProvider(pipeline_config)
pipeline_schema = pipeline_provider.load_schema()

print(f'Pipeline schema ready: {pipeline_schema.schema_id}')
print(f'  Extract: {pipeline_schema.extract}')
print(f'  Transform: {pipeline_schema.transform}')
print(f'  Load: {pipeline_schema.load}')


## Part 3: Schema Validation

ETLSchema is a Pydantic model — it validates structure at construction.


In [ ]:
# Valid schema construction (all 3 phases required)
from document_graph.schema.etl_schema_model import (
    ETLSchema, ExtractConfig, TransformConfig, LoadConfig,
    ChunkingConfig, NormalizeConfig, EntityExtractionConfig,
    MetadataMapping, NodeDefinition, RelationshipDefinition
)

valid = ETLSchema(
    schema_id="test-valid-v1",
    description="A valid test schema",
    extract=ExtractConfig(source_type="csv", file_type="csv"),
    transform=TransformConfig(
        chunking=ChunkingConfig(strategy="fixed_length", min_length=100),
        metadata_mapping=MetadataMapping(),
        entity_extraction=EntityExtractionConfig(method="ner"),
        normalize=NormalizeConfig()
    ),
    load=LoadConfig(
        document_node=NodeDefinition(type="Document", fields=["id", "name", "email"]),
        section_node=NodeDefinition(type="Row", fields=["id", "name", "email"]),
        relationships=[RelationshipDefinition(type="contains", source="doc_id", target="row_id")]
    )
)
print(f"Valid schema: {valid.schema_id}")
print(f"  Extract: {valid.extract.source_type}")
print(f"  Load fields: {valid.load.document_node.fields}")


In [ ]:
# Invalid schema — missing required fields
from pydantic import ValidationError

try:
    bad = ETLSchema(schema_id="incomplete")  # missing transform + load
except ValidationError as e:
    print(f"Validation caught {e.error_count()} error(s):")
    for err in e.errors():
        loc = err["loc"]
        msg = err["msg"]
        print(f"  {loc}: {msg}")


In [ ]:
# Schema round-trip: model → JSON → model
exported = valid.model_dump(exclude_none=True)
reimported = ETLSchema(**exported)
assert reimported.schema_id == valid.schema_id
assert reimported.load.document_node.fields == valid.load.document_node.fields
print(f"Round-trip OK: {reimported.schema_id}")
print(f"  JSON size: {len(json.dumps(exported))} bytes")


## Summary

| Provider | Source | Use Case |
|----------|--------|----------|
| `CSVSchemaProvider` | Local CSV file | Auto-discover schema from tabular data |
| `JSONSchemaProvider` | Local JSON file | Load pre-defined schemas |
| `S3SchemaProvider` | S3 bucket | Shared schemas across environments |
| `StaticSchemaProvider` | Code (dict) | Programmatic schema definition |
| `FileSchemaProvider` | Any local file | Generic file-based loading |
| `SchemaProviderFactory` | Config dict | Unified creation interface |

All providers return `ETLSchema` (Pydantic model) — validated, serializable, type-safe.
